# **COMP 2211 Introduction to Artificial Intelligence** #
## Lab 2: Naïve Bayes Classifier ##

This lab uses a bank credit card customer dataset where you are trying to predict is the customer’s attrition status, which indicates whether the customer is still actively using the credit card. The label “Existing Customer” means the customer is still a current user of the card, while “Attrited Customer” means the customer has stopped using the card and is no longer active.

To predict whether a customer is an "Existing Customer" or "Attrited Customer", you are given six features for each customer. Three of them are Categorical variables, and the other three are numerical variables. The information about each variable is given below:

Categorical Variables:
* Gender: F, M
* Marital_Status: Divorced, Married, Single, Unknown
* Income_Category: Less than $40K, $40K - $60K, $60K - $80K, $80K - $120K, $120K +, Unknown

Numerical Variable:
* Customer_age: min: 26, max: 65, mean: 46.68
* Credit_Limit: min: 1438, max: 34516, mean: 6512.19
* Total_Revolving_Bal: min: 0, max: 2517, mean: 1095.74

The beliefs or prediction values are also categorical.
* Attrition_Flag: Attrited Customer, Existing Customer

Run the following code to get more information about the dataset. You can modify the code to view additional information as well.

In [1]:
if __name__ == '__main__':
  import numpy as np
  import pandas as pd

  x_train = pd.read_csv('x_train_pub.csv')
  y_train = pd.read_csv('y_train_pub.csv')
  x_test = pd.read_csv('x_test_pub.csv')
  y_test = pd.read_csv('y_test_pub.csv')

  print("x_train info:")
  print(x_train.head())
  print("\ny_train info:")
  print(y_train.head())



x_train info:
  Gender Marital_Status Income_Category  Customer_Age  Credit_Limit  \
0      F         Single         Unknown            51        8365.0   
1      F         Single  Less than $40K            56        2846.0   
2      F         Single  Less than $40K            49        7789.0   
3      F         Single  Less than $40K            44        1438.3   
4      F        Married         Unknown            48        2220.0   

   Total_Revolving_Bal  
0                 1504  
1                    0  
2                  957  
3                  170  
4                    0  

y_train info:
      Attrition_Flag
0  Existing Customer
1  Attrited Customer
2  Existing Customer
3  Attrited Customer
4  Attrited Customer


### Task 1: Change Categorical data into numerical value

For this task, you will convert the categorical features and beliefs in the dataset into numeric codes to prepare the data for a Naive Bayes classifier. Specifically, they should map each category to an integer, such as:

Attrition_Flag: Attrited Customer → 0, Existing Customer → 1

Gender: F → 0, M → 1

Marital_Status: Divorced → 0, Married → 1, Single → 2, Unknown → 3

Income_Category: Less than $40K → 0, $40K - $60K → 1, $60K - $80K → 2, $80K - $120K → 3, $120K + → 4, Unknown → 5




In [2]:
import numpy as np
import pandas as pd

def pandas_to_numpy_features(df):
  """
  Converts the Pandas dataframe of features to a NumPy dataframe using the conversion rules stated above
  Args:
    df: the Pandas dataframe, with shape (num samples, 6)
  Returns:
    A NumPy array of shape (num samples, 6)
  """

  data = df.values  # converts the pandas dataframe to a NumPy array

  # Ensure we have a mutable string/number array
  data = data.astype(object)

  #TODO part 1 start
  mappings = {
        "Attrition_Flag": {
            "Attrited Customer": 0,
            "Existing Customer": 1
        },
        "Gender": {
            "F": 0,
            "M": 1
        },
        "Marital_Status": {
            "Divorced": 0,
            "Married": 1,
            "Single": 2,
            "Unknown": 3
        },
        "Income_Category": {
            "Less than $40K": 0,
            "$40K - $60K": 1,
            "$60K - $80K": 2,
            "$80K - $120K": 3,
            "$120K +": 4,
            "Unknown": 5
        }
    }
    
  #that is how normal people do
  df_encoded = df.replace(mappings)
  return df_encoded.to_numpy(dtype=int)

  # TODO part 1 end

  # return data.astype(int) # Finally convert everything to int

def pandas_to_numpy_belief(df):
  """
  Converts the Pandas dataframe of beliefs to a NumPy dataframe
  Args:
    df: the Pandas dataframe, with shape (num samples, 1)
  Returns:
    A NumPy array of shape (num samples, 1)
  """
  data = df.values  # converts the pandas dataframe to a NumPy array

  # Ensure we have a mutable string/number array
  data = data.astype(object)

  #we dont need the stuff above

  # TODO part 2 start
  belief_mapping = {
        "Attrited Customer": 0,
        "Existing Customer": 1
    }
    
  df_encoded = df.replace(belief_mapping)
  return df_encoded.to_numpy(dtype=int)

  # TODO part 2 end

  # return data.astype(int) # Finally convert everything to int

In [3]:
if __name__ == '__main__':
  # Verify Task 1
  _x_proc = pandas_to_numpy_features(x_train)
  _y_proc = pandas_to_numpy_belief(y_train)
  _x_test_proc = pandas_to_numpy_features(x_test)
  _y_test_proc = pandas_to_numpy_belief(y_test)

  _y_flat = _y_proc.reshape(-1)
  _y_test_flat = _y_test_proc.reshape(-1)

  print("x_train shape:", _x_proc.shape)    # (4051, 6)
  print("y_train shape:", _y_flat.shape)     # (4051,)
  print("x_test shape:", _x_test_proc.shape) # (1013, 6)
  print("y_test shape:", _y_test_flat.shape)  # (1013,)
  print(_x_test_proc)
  print(_y_proc)

x_train shape: (4051, 6)
y_train shape: (4051,)
x_test shape: (1013, 6)
y_test shape: (1013,)
[[    1     1     2    41 17277  2151]
 [    1     3     4    56 34516  2156]
 [    0     2     5    39 33565  1627]
 ...
 [    0     1     0    44  5409     0]
 [    1     3     1    30  5281     0]
 [    0     1     0    43 10388  1961]]
[[1]
 [0]
 [1]
 ...
 [1]
 [0]
 [1]]


/var/folders/jc/8r_cxn9j5fncd5rm5gd545q80000gn/T/ipykernel_1192/757869695.py:45: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded = df.replace(mappings)
/var/folders/jc/8r_cxn9j5fncd5rm5gd545q80000gn/T/ipykernel_1192/757869695.py:73: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded = df.replace(belief_mapping)


## Task 2: Estimate prior probabilities

Here is the general form of the equation for obtaining the prior probabilities:

$$P(belief=j)= \frac{\text{count samples}(belief=j)}{\text{train size}}$$

For this lab activity, you do **not** need to apply Laplace smoothing on the prior probabilities.

Although there are only two classes here, note that your solution should be applicable to 2 or more classes.

This means you should **not** make a solution like: $prior_{(y=0)} = 1 - prior_{(y=1)}$

**Todo:** Implement the *estimate_priors()* function that calculates the prior probabilities for each belief.

**Some hints:**
* Useful Numpy functions include `numpy.sum()`, `numpy.arange()`, `numpy.expand_dims()` or `array.reshape()`,...
* You can apply this masking trick to index the samples with a particular belief:
  * Let's say you have a dataset with beliefs `y = np.array([0, 1, 2, 2, 1, 1])`
  * And you want to count the number of samples with belief `1`
  * Then you can get the indices of the samples with the code `y == 1`, which results in an array of boolean values `array([False,  True, False, False,  True,  True])`
  * After applying masking, you can take the sum of these boolean values to count the number of `True` values.
  

In [4]:
# Task 2: Estimate priors

def estimate_priors(y_train):
  # y_train: a 1D numpy array with shape (num_train_samples, )

  # TODO start
  # class_number=np.unique(y_train).size
  size=y_train.size
  class_priors=np.unique(y_train)
  re=class_priors.reshape(-1,1)
  pro=re-y_train

  pro=np.sum(pro==0,axis=1)/size
  class_priors=pro

  # TODO end

  return class_priors
  # class priors: a 1D numpy array with shape (num_classes, )

In [5]:
if __name__ == '__main__':
  # Verify Task 2
  _y_flat = pandas_to_numpy_belief(y_train).reshape(-1)
  _priors = estimate_priors(_y_flat)
  print('Priors:', _priors)
  # Expected: [0.19451987 0.80548013]


Priors: [0.19451987 0.80548013]


/var/folders/jc/8r_cxn9j5fncd5rm5gd545q80000gn/T/ipykernel_1192/757869695.py:73: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded = df.replace(belief_mapping)


## Task 3: Estimate the likelihood of a categorical feature

Here is the general form for calculating the likelihood of a categorical feature:
$$P(feature_i=yes|belief=j)=\frac{\text{count samples}(feature_i=yes \& belief=j)+\alpha}{\text{count samples}(belief=j)+\alpha \cdot\text{num feature classes}}$$

**Todo:**
* Implement a `cat_likelihood` function that computes the likelihood for a **single** categorical feature.
* The function should return a numpy array with size (num_test_samples, num_classes) where the values correspond to the category type of that feature. Here, num_classes is the number of kinds of beliefs.

**Approach — This task can be divided into two parts:**

**Part A: Compute the likelihood table.** Build a 2D array of shape `(num_feature_categories, num_classes)`, where each entry stores the smoothed probability of a feature category given a class. For example, if the feature "Gender" has 2 categories (F=0, M=1) and there are 2 classes, the table would look like:

| | Class 0 | Class 1 |
|---|---|---|
| **Gender=0 (F)** | P(F\|class 0) | P(F\|class 1) |
| **Gender=1 (M)** | P(M\|class 0) | P(M\|class 1) |

Each entry is computed using the Laplace-smoothed formula above.

**Part B: Look up the likelihood for each test sample.** Use the feature values of the test data to index into the likelihood table from Part A, producing a 2D array of shape `(num_test_samples, num_classes)`.

**Some hints:**
* Notice that we applied Laplace smoothing.
* You can apply masking to build indicator matrices for both class labels and feature categories, then use matrix multiplication (`numpy.dot()`) to efficiently count co-occurrences.
* Useful Numpy functions include `numpy.max()`, `numpy.arange()`, `numpy.dot()`, `numpy.matmul()`, ...


In [6]:
# Task 3: Likelihood for categorical feature

def cat_likelihood(X_train_feature, y_train, X_test_feature, alpha = 0.1):
  # X_train_feature: a 1D numpy array with shape (num_train_samples, )
  # y_train: a 1D numpy array with shape (num_train_samples, )
  # X_test_feature: a 1D numpy array with shape (num_test_samples, )

  # TODO start
  #  Get unique values (categories/classes)
    feature_cats = np.unique(X_train_feature)  #[f,f,m,f] ->[0,0,1,0]->[0,1]
    classes = np.unique(y_train)               #[1,0,1,0] ->[0,1] ytrain is class
    num_feature_cats = len(feature_cats) 
    num_classes = len(classes)                 

    #   Initialize likelihood table
    likelihood_table = np.zeros((num_feature_cats, num_classes))

    #   Compute smoothed likelihood for each (feature_cat, class) pair
    for class_idx, cls in enumerate(classes):
        # 1. Count total samples in this class (denominator base)
        class_mask = (y_train == cls)          # Boolean mask for samples in class `cls`
        total_in_class = np.sum(class_mask)    # Total samples in this class

        # 2. Denominator with Laplace smoothing: total_in_class + alpha * num_feature_cats
        denominator = total_in_class + alpha * num_feature_cats

        # 3. For each feature category, compute numerator + smoothed probability
        for cat_idx, cat in enumerate(feature_cats):
            # Count samples where feature == cat AND class == cls
            count_cooccur = np.sum((X_train_feature == cat) & class_mask)
            
            numerator = count_cooccur + alpha
            
            # Store smoothed probability in the table
            likelihood_table[cat_idx, class_idx] = numerator / denominator

    # Part B: Look up likelihoods for test samples

    #  1:  a mapping from feature category to its index for fast lookup.
    cat_to_idx = {cat: idx for idx, cat in enumerate(feature_cats)}
    
    #category : index, since so we can get the row from the index when looping through test_feature
    
    #   2: For each test sample, get its feature category → find index → get likelihoods

    feat_likelihood = np.zeros((len(X_test_feature), num_classes))
    
    for test_idx, test_cat in enumerate(X_test_feature):
        # Get the index of the test feature category in our feature_cats list
        cat_idx = cat_to_idx[test_cat]
        # Look up the likelihoods for this category across all classes
        feat_likelihood[test_idx, :] = likelihood_table[cat_idx, :]

    # return feat_likelihood


  # TODO end

    return feat_likelihood
  # feat_likelihood: a 2D numpy array with shape (num_test_samples, num_classes)


In [7]:
if __name__ == '__main__':
  # Verify Task 3
  _x_proc = pandas_to_numpy_features(x_train)
  _x_test_proc = pandas_to_numpy_features(x_test)
  _y_flat = pandas_to_numpy_belief(y_train).reshape(-1)
  _lk = cat_likelihood(_x_proc[:, 0], _y_flat, _x_test_proc[:, 0])
  print('Likelihood for "Gender" (first 5 test samples):')
  print(_lk[:5])
  # Expected:
  # [[0.29573712 0.33007477]
  #  [0.29573712 0.33007477]
  #  [0.70426288 0.66992523]
  #  [0.70426288 0.66992523]
  #  [0.29573712 0.33007477]]


Likelihood for "Gender" (first 5 test samples):
[[0.29573712 0.33007477]
 [0.29573712 0.33007477]
 [0.70426288 0.66992523]
 [0.70426288 0.66992523]
 [0.29573712 0.33007477]]


/var/folders/jc/8r_cxn9j5fncd5rm5gd545q80000gn/T/ipykernel_1192/757869695.py:45: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded = df.replace(mappings)
/var/folders/jc/8r_cxn9j5fncd5rm5gd545q80000gn/T/ipykernel_1192/757869695.py:73: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded = df.replace(belief_mapping)


## Task 4: Estimate the likelihood of a numerical feature

Here is the general form for calculating the likelihood of a numerical variable:
$$P(feature_i=x|belief=j)=\frac{1}{\sqrt{2 \pi \sigma_j^2}} \exp{ \left( -\frac{(x-\mu_j)^2}{2 \sigma_j^2} \right)}$$

**Todo:**
In general, implement a `num_likelihood` function that computes the likelihood for a **single** numerical feature.
1. Implement a `feature_mean` function that calculates the mean feature value $\mu_j$ for samples according to their belief $j$.
2. Implement a `feature_var` function that calculates the variance of feature value $\sigma_j^2$ for samples according to their belief $j$.

$$\sigma_j^2 = \frac{1}{n_j} \sum_{i=1}^{n_j}{(x_i - \mu_j)^2}$$

3. Consolidate these functions in a `num_likelihood` function that calculates the likelihood value.
The function should return a numpy array with size (num_test_samples, num_classes) where the values correspond to the value of that feature.

**Some hints:**
* Useful numpy functions include many of the functions suggested in previous tasks, `numpy.square()`, `numpy.exp()`, `numpy.sqrt()`, ...



In [8]:
def num_likelihood(X_train_feature, y_train, X_test_feature):
  # X_train_feature: a 1D numpy array with shape (num_train_samples, )
  # y_train: a 1D numpy array with shape (num_train_samples, )
  
  # X_test_feature: a 1D numpy array with shape (num_test_samples, )
  def feature_mean(X_train_feature, y_train):
    # TODO Part 1 start
    classes=np.unique(y_train)
    mean = classes

    num_classes = len(classes)
    
    # Vectorized mask: (num_classes, num_train_samples)
    # class_masks[j, i] = True if y_train[i] == classes[j]
    class_masks = y_train == classes[:, np.newaxis]
    
    # Compute mean for each class (sum feature values where mask=True / count of True)
    # np.sum with axis=1: sum over samples for 'each class'
    # np.count_nonzero with axis=1: count samples 'per class'
    mean = np.sum(X_train_feature * class_masks, axis=1) / np.count_nonzero(class_masks, axis=1)
      
#   X_train_feature * class_masks = np.array([
#     [30, 35, 40, 45, 50, 0, 0, 0, 0, 0],  # Class 0: only first 5 Age values (others 0)
#     [0, 0, 0, 0, 0, 25, 30, 35, 40, 45]   # Class 1: only last 5 Age values (others 0)
# ])

# # Shape: (2, 10) (same as class_masks)

    # TODO Part 1 end
    return mean
    # mean: a 1D numpy array with shape (num_classes, )

  def feature_var(X_train_feature, y_train):
    # TODO Part 2 start
    classes = np.unique(y_train)
    num_classes = len(classes)
    
    # 1: Get mean (vectorized)
    mu = feature_mean(X_train_feature, y_train)
    
    # 2: Vectorized mask 
    class_masks = y_train == classes[:, np.newaxis]
    
    # Vectorized variance calculation (matches task formula: 1/n_j * sum((x_i - μ_j)²))
    # Broadcast mu to (num_classes, num_train_samples) to subtract from feature values
    squared_diff = (X_train_feature - mu[:, np.newaxis]) ** 2

    # Sum squared differences per class, divide by number of samples per class
    var = np.sum(squared_diff * class_masks, axis=1) / np.count_nonzero(class_masks, axis=1)
      
    # TODO Part 2 end
    return var
    # var: a 1D numpy array with shape (num_classes, )

  mean = feature_mean(X_train_feature, y_train)
  var = feature_var(X_train_feature, y_train)

  # TODO Part 3 start
  x = X_test_feature.reshape(-1, 1)
  exponent = -(x - mean) ** 2 / (2 * var)
  coeff = 1 / np.sqrt(2 * np.pi * var)
  feat_likelihood = coeff * np.exp(exponent)

  # TODO Part 3 end

  return feat_likelihood
  # feat_likelihood: a 2D numpy array with shape (num_test_samples, num_classes)


In [9]:
if __name__ == '__main__':
  # Verify Task 4
  _x_proc = pandas_to_numpy_features(x_train)
  _x_test_proc = pandas_to_numpy_features(x_test)
  _y_flat = pandas_to_numpy_belief(y_train).reshape(-1)
  _lk = num_likelihood(_x_proc[:, 3], _y_flat, _x_test_proc[:, 3])
  print('Likelihood for "Customer_Age" (first 5 test samples):')
  print(_lk[:5])
  # Expected:
  # [[0.04243951 0.04091782]
  #  [0.02216243 0.02340841]
  #  [0.03283806 0.03063091]
  #  [0.05281076 0.05532777]
  #  [0.05563784 0.05647077]]


Likelihood for "Customer_Age" (first 5 test samples):
[[0.04243951 0.04091782]
 [0.02216243 0.02340841]
 [0.03283806 0.03063091]
 [0.05281076 0.05532777]
 [0.05563784 0.05647077]]


/var/folders/jc/8r_cxn9j5fncd5rm5gd545q80000gn/T/ipykernel_1192/757869695.py:45: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded = df.replace(mappings)
/var/folders/jc/8r_cxn9j5fncd5rm5gd545q80000gn/T/ipykernel_1192/757869695.py:73: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded = df.replace(belief_mapping)


## Task 5: Prediction

The predicted class is based on the posterior probability. Here is the general form of the posterior probability:

$$P(belief=j|features)=\frac{P(belief=j)P(f_1|belief=j)P(f_2|belief=j)P(f_3|belief=j)\ldots P(f_m|belief=j)}{\sum_{i=1}^n P(belief=i)P(f_1|belief=i)P(f_2|belief=i)P(f_3|belief=i)\ldots P(f_m|belief=i)}$$

However, if we are only interested in the predicted class and not the exact posterior probability calculation, we __don't__ need to worry about the denominator (**Marginal Probability**), as the denominator is the same for all posterior probability values regardless of the belief. Hence, the prediction can be made by taking the argmax of the numerator:
$$\hat{y} = argmax_{j} P(belief=j)P(f_1|belief=j)P(f_2|belief=j)P(f_3|belief=j)\ldots P(f_m|belief=j)$$



In addition, multiplying several probability values by each other can result in floating-point underflow. So instead of using the equation above, it is recommended to base the prediction on the $\log$ of the numerator.

$$\hat{y} = argmax_{j}  \log P(belief=j) + \log P(f_1|belief=j) + \ldots + \log P(f_m|belief=j) $$

**Todo:**
* Implement a `predict` function that makes predictions for the test dataset.
* Use the functions you created in Tasks 2-4 in the `predict` function.

**Some hints:**
* It is recommended to use a **for loop** to iterate over each feature and accumulate the log-likelihood contributions.
* **Reminder:** Nested loops and recursive function calls are **not** allowed. However, a single (stand-alone) for loop is fine. Within the loop body, use **NumPy vectorization** to process all samples at once for each feature.


In [10]:
def predict(X_train, y_train, X_test, feat_type_list):
  # X_train: a 2D numpy array with shape (num_train_samples, num_features)
  # y_train: a 1D numpy array with shape (num_train_samples, )
  # X_test: a 2D numpy array with shape (num_test_samples, num_features)
  # feat_type_array: a list of feature types in string form in the order of features in X_train and X_test
    # For example: feat_type_array = [ 'cat', 'cat', 'cat', 'num', 'num', 'num']

  # TODO start
  classes = np.unique(y_train)
  class_counts = np.bincount(y_train)   #count for each class
  prior = class_counts / len(y_train)    # [P(0), P(1)]
  log_prior = np.log(prior)              # [ log(P(0)), log(P(1)) ]

#  初始化总分：每行是 [score0, score1]
  n_test = X_test.shape[0]
  n_classes = len(classes)
  total = np.zeros((n_test, n_classes))

  for i in range(len(feat_type_list)):
      f_train = X_train[:, i]
      f_test  = X_test[:, i]
      t = feat_type_list[i]

      if t == 'cat':
          lik = cat_likelihood(f_train, y_train, f_test)
      else:
          lik = num_likelihood(f_train, y_train, f_test)

      # 把这个特征的 log 分数加进去
      total += np.log(lik + 1e-10)

      # 4. 加先验，比大小
  total += log_prior
  test_predict = np.argmax(total, axis=1)
  # TODO end

  return test_predict
  # test_predict: a 1D numpy array with shape (num_test_samples, )


In [11]:
if __name__ == '__main__':
  # Verify Task 5
  _x_proc = pandas_to_numpy_features(x_train)
  _x_test_proc = pandas_to_numpy_features(x_test)
  _y_flat = pandas_to_numpy_belief(y_train).reshape(-1)
  _y_test_flat = pandas_to_numpy_belief(y_test).reshape(-1)
  _pred = predict(_x_proc, _y_flat, _x_test_proc, feat_type_list=['cat', 'cat', 'cat', 'num', 'num', 'num'])
  _acc = np.sum(_pred == _y_test_flat) / _y_test_flat.shape[0]
  print('Accuracy:', _acc)
  # Expected: 0.7482724580454096


Accuracy: 0.7482724580454096


/var/folders/jc/8r_cxn9j5fncd5rm5gd545q80000gn/T/ipykernel_1192/757869695.py:45: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded = df.replace(mappings)
/var/folders/jc/8r_cxn9j5fncd5rm5gd545q80000gn/T/ipykernel_1192/757869695.py:73: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded = df.replace(belief_mapping)
